This is the notebook to run inference on trained model using the `CoppeliaSim` robot simulator. Follow the below steps for running this notebook

1. Download and install CoppeliaSim from here: [https://www.coppeliarobotics.com/](https://www.coppeliarobotics.com/)
2. Open the scene file `\simulation\sim_envs\coppeliasim_scene_for_spraying.ttt` in the simulator software (must do it before running the notebook).
3. Run all the cells in the notebook (there is an assertion error just to prevent running the notebook when using the run all option. Please continue running below cells if you get assertion error.)

In [1]:
# Hyperparameters and paths
num_robots = 2 # Valid options: 2-5
exp_alg = 'CrossQ' # Valid options: A2C, ARS, CrossQ, PPO, TQC, TRPO
exp_set = 1 # Valid options: 1-10

In [2]:
import json
import itertools
import numpy as np
import copy
import pygame
import gymnasium as gym
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
np.random.seed(33) # seeding

if exp_alg.lower() == 'crossq':
    from sb3_contrib import CrossQ as model_loader 
elif exp_alg.lower() == 'trpo':
    from sb3_contrib import TRPO as model_loader

max_steps = 1000
if num_robots > 3:
    json_path = rf'..\exp_sets\new_2026_sets_five.json'
else:
    json_path = rf'..\exp_sets\new_2026_sets_five.json'
trained_model_path = rf"..\training_default_logs\{num_robots}_robots\weights\env{exp_set}_{exp_alg}.zip"

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# Function to check if a point is inside a polygon (Ray-casting algorithm)
def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to return minimum distance in a list of points
def compute_min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1-p2)
        dists.append(dist)
    return float(np.min(dists))

# Load experiment json file
def load_experiment_dict_json(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    for set_name, cfg in data.items():
        # Convert field to list of tuples
        cfg["field"] = [tuple(p) for p in cfg["field"]]
        # Convert init_positions to NumPy arrays
        cfg["init_positions"] = [np.array(p, dtype=float) for p in cfg["init_positions"]]
        # Convert infected_locations to set of tuples
        cfg["infected_locations"] = [tuple(p) for p in cfg["infected_locations"]]
    return data

Make the multi-agent class:

In [4]:
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}    
    def __init__(self, field_info, render_mode=None, wind_par=[0,0], num_robots=3, render_scale=10):
        super().__init__()

        # Screen, rendering (simulation) and field parameters
        self.field_info = copy.deepcopy(field_info)
        self.render_scale = render_scale # Scaling factor for rendering only 
        self.poly_vertices = self.field_info['field']  # Physics coordinates
        self.xs, self.ys = zip(*self.field_info['field']) # x and y values of the vertices of the polygonal field
        self.WIDTH, self.HEIGHT = max(self.xs) + 10, max(self.ys) + 10 # Boundary above the max values
        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check valid render modes
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        # Robot parameters
        self.num_robots = num_robots
        self.robot_size = 2
        self.mass = 1.0
        self.thrust_power = 0.5 # Force applied per action
        self.max_speed = 5
        self.min_speed = -5
        self.init_robot_positions = np.array(self.field_info['init_positions'])[:self.num_robots]
        self.init_robot_capacities = np.array(self.field_info['robot_capacities'][:self.num_robots], dtype=np.float32)
        self.max_capacity = np.max(self.init_robot_capacities)
        self.wind_f_a, self.wind_beta_a = wind_par # Wind parameters: magnitude and angle

        # Infected locations with levels
        self.init_infected_locations = [(np.array([x, y]), level) for x, y, level in self.field_info['infected_locations']]
        self.max_infection_level = max(lvl for _, lvl in self.init_infected_locations)
        self.infected_radius = 5
        self.spray_sigma = self.infected_radius / 2.0
        self.n_infected = len(self.init_infected_locations)
        
        # Action space (ax, ay, spray) and Observation space (x, y, vx, vy, inf_loc): position and velocity for each robot and infected location
        self.action_space = gym.spaces.Box(
            low=np.array([[-1.0, -1.0, 0.0]] * self.num_robots),
            high=np.array([[1.0, 1.0, 1.0]] * self.num_robots), dtype=np.float32
        )
        self.observation_space = gym.spaces.Box(
            low=np.concatenate((
                np.zeros(self.num_robots * 2),                # (x, y) co-ordinates of each robot
                np.full(self.num_robots * 2, self.min_speed), # (v_x, v_y) velocities along x and y axis of each robot
                np.zeros(self.num_robots), # robot spraying capacities
                np.zeros(self.n_infected)  # infection levels of each infected locations
            )),
            high=np.concatenate((
                np.tile([self.WIDTH, self.HEIGHT], self.num_robots), # (x, y) co-ordinates of each robot
                np.full(self.num_robots * 2, self.max_speed), # (v_x, v_y) velocities along x and y axis of each robot
                np.full(self.num_robots, self.max_capacity), # Full capacity for all robots
                np.ones(self.n_infected) # infection levels of each infected locations
            )), dtype=np.float32
        )
        
        self.reset() # Reset environment and start

    def _get_obs(self):
        infection_levels = np.array([lvl / self.max_infection_level 
                                     for _, lvl in self.infected_locations], dtype=np.float32) # Normalized infection levels for infected locations
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(),
                                self.robot_capacities, infection_levels), dtype=np.float32)
        return state

    def reset(self, seed=None, options={}):
        super().reset(seed=seed)
        self.robot_positions = copy.deepcopy(self.init_robot_positions) # Initial robot locations
        self.robot_velocities = np.zeros((self.num_robots, 2), dtype=np.float32) # Initial velocities of each robot (zero)
        self.robot_capacities = copy.deepcopy(self.init_robot_capacities) # Initial robot capacities
        self.infected_locations = copy.deepcopy(self.init_infected_locations) # Initial infected locations
        self.visited = set() # Keep track of visited states
        self.last_actions = np.zeros((self.num_robots, 3), dtype=np.float32)  # Keep track of actions taken
        info = {f'robot{i}': {'position': self.robot_positions[i], 'capacity': self.robot_capacities[i]}
                    for i in range(self.num_robots)}
        obs = self._get_obs() # Get current state
        return obs, info

    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.last_actions = actions.copy()
        for i in range(self.num_robots): # For every robot
            ax, ay, spray = actions[i] # Get the position and spray levels for the robot

            # ----- Gaussian area spray -----
            if spray > 0 and self.robot_capacities[i] > 0: # Check spraying level
                for idx, (loc, level) in enumerate(self.infected_locations): # Check each infected location
                    dist = np.linalg.norm(self.robot_positions[i] - loc) # Distance between robot and infected location
                    if dist <= self.infected_radius and level > 0: # If the distance is within infected radius
                        w = np.exp(-(dist ** 2) / (2 * self.spray_sigma ** 2)) # Gaussian spray value
                        applied = min(spray * w, self.robot_capacities[i], level) # Take the minimum of (spray_amount, robot_capacity, infection level)
                        self.robot_capacities[i] -= applied # Reduce the robot capacity
                        self.infected_locations[idx] = (loc, level - applied) # Updated the spraying level
                        rewards += 2000.0 * applied

            # Movement by robot dynamics
            ax, ay = ax * self.thrust_power, ay * self.thrust_power
            self.robot_velocities[i, 0] += ax / self.mass + self.wind_f_a * np.cos(np.radians(self.wind_beta_a))
            self.robot_velocities[i, 1] += ay / self.mass + self.wind_f_a * np.sin(np.radians(self.wind_beta_a))
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Check new position and update visiteds states
            new_position = self.robot_positions[i] + self.robot_velocities[i]
            if not is_inside_polygon(new_position, self.poly_vertices): # If new position is outside the field
                rewards -= 10000 # Big negative reward for going outside the field
                self.robot_velocities[i][:] = 0
            else:
                self.robot_positions[i] = new_position
            pos_key = tuple(np.round(self.robot_positions[i], 1))
            if pos_key in self.visited: # If current location is visited previously
                rewards -= 100 # Small negative reward for visiting same state
            else:
                rewards -= 10 # Very small negative reward for visiting new state
            self.visited.add(pos_key)

        if all(level <= 0.01 for _, level in self.infected_locations): # If all infected locations are cleared
            rewards += 100000 # Very big reward for visiting all infected locations and terminate
            terminated = True

        if self.num_robots > 1:
            min_dist_between_robots = compute_min_dist(self.robot_positions)
            if min_dist_between_robots < self.robot_size: # Check if any collisions occured
                rewards = -100000 # Very big negative reward for collisions and terminate
                terminated = True

        info = {f'robot{i}': {'position': self.robot_positions[i], 'capacity': self.robot_capacities[i]}
                    for i in range(self.num_robots)}

        obs = self._get_obs()
        return obs, rewards, terminated, truncated, info

    def render(self):
        if self.screen is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode(
                (self.WIDTH * self.render_scale, self.HEIGHT * self.render_scale)
            )
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True

        self.screen.fill((255, 255, 255))
        colors = [
            (255, 0, 0), (0, 255, 0), (0, 0, 255),
            (255, 128, 0), (128, 0, 255), (255, 0, 255)
        ] # Red, Green, Blue, Orange (or Dark Yellow), Purple (or Violet), and Magenta (or Fuchsia)
        pix_size = 10

        # ---------- Draw field polygon ----------
        scaled_poly = [(x*self.render_scale, y*self.render_scale) for x, y in self.poly_vertices]
        pygame.draw.polygon(surface=self.screen, color=(255, 255, 0), points=scaled_poly)

        # ---------- Draw visited points ----------
        for point in self.visited:
            scaled_point = (int(point[0]*self.render_scale), int(point[1]*self.render_scale))
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), scaled_point, pix_size//2)

        # ---------- Draw Gaussian spray based on ACTION ----------
        spray_radius_px = int(self.infected_radius * self.render_scale)
        sigma_px = self.spray_sigma * self.render_scale

        for i in range(self.num_robots):
            spray = float(self.last_actions[i][2])
            if spray <= 0.0:
                continue
            center = (
                int(self.robot_positions[i][0] * self.render_scale),
                int(self.robot_positions[i][1] * self.render_scale)
            )
            spray_surface = pygame.Surface(
                (2 * spray_radius_px, 2 * spray_radius_px),
                pygame.SRCALPHA
            ).convert_alpha()  # ✅ IMPORTANT
            peak_alpha = int(200 * np.clip(spray, 0.0, 1.0))
            
            for r in range(spray_radius_px, 0, -1):
                gaussian = np.exp(-(r ** 2) / (2 * sigma_px ** 2))
                alpha = int(peak_alpha * gaussian)
                if alpha < 2:
                    continue
                pygame.draw.circle(
                    spray_surface,
                    (colors[i][0], colors[i][1], colors[i][2], alpha),
                    (spray_radius_px, spray_radius_px),
                    r
                )

            self.screen.blit(
                spray_surface,
                (center[0] - spray_radius_px, center[1] - spray_radius_px)
            )

        # ---------- Draw robots ----------
        for i in range(self.num_robots):
            scaled_pos = (int(self.robot_positions[i][0]*self.render_scale), int(self.robot_positions[i][1]*self.render_scale))
            pygame.draw.circle(self.screen, colors[i], scaled_pos, pix_size//2)

        # ---------- Draw infected locations ----------
        for loc, level in self.infected_locations:
            if level < 0.01:
                continue
            intensity = int(255 * (1- min(level/5.0, 1.0)))
            scaled_loc = (int(loc[0]*self.render_scale), int(loc[1]*self.render_scale))
            pygame.draw.circle(self.screen, (0, intensity, 255), scaled_loc, 6)

        pygame.display.flip()
        self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=max_steps)

# Inference

In [5]:
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

class Drone_simulator:
    def __init__(self, gym_env, scaling_factor=5, height=0.35, num_robots=3):
        self.scaling_factor = scaling_factor
        self.polygon = gym_env.unwrapped.poly_vertices
        self.scaled_polygon = [(x/scaling_factor, y/scaling_factor) for (x,y) in self.polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.weed_locations=[loc for loc, _ in gym_env.unwrapped.init_infected_locations]
        self.height = height
        self.num_robots = num_robots
        self.max_robots = 5

        # cache script handles (important!)        
        self.nozzle_scripts = []
        for i in range(num_robots):
            # Directly get the Lua script handle
            script_path = f"/Quadcopter[{i}]/PaintNozzle/Script"
            script_handle = sim.getObject(script_path)
            self.nozzle_scripts.append(script_handle)
        
        # Drawing handle
        self.field_drawing = None

        # ---- cache quadcopters & initial states ----
        self.all_drones = []
        self.initial_positions = {}
        i = 0
        while True:
            h = sim.getObject(f"/Quadcopter[{i}]", {'noError': True})
            if h == -1:
                break
            self.all_drones.append(h)
            self.initial_positions[h] = sim.getObjectPosition(h, -1)
            i += 1

    # ------------------------------------------------

    def start_simulation(self):
        sim.startSimulation()
        print("Simulation started")

    def stop_simulation(self):
        sim.stopSimulation()
        while sim.getSimulationState() != sim.simulation_stopped:
            sim.step()
        
        # ---- cleanup runtime artifacts ----
        self.stop_spraying()
        self.clear_field()

        # ---- restore robots ----
        for drone, pos in self.initial_positions.items():
            sim.setObjectPosition(drone, -1, pos)
            sim.setModelProperty(drone, 0)
        print("Simulation reset completed")

    # ------------------------------------------------

    def draw_field(self):
        white = [255, 255, 255]
        self.field_drawing = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, white)

        for i in range(len(self.rounded_polygon) - 1):
            p1 = self.rounded_polygon[i]
            p2 = self.rounded_polygon[i+1]
            line = [
                p1[0], p1[1], 0.1,
                p2[0], p2[1], 0.1     # drawing at height 0.1
            ]
            sim.addDrawingObjectItem(self.field_drawing, line)
    
    def clear_field(self):
        if self.field_drawing is not None:
            sim.removeDrawingObject(self.field_drawing)
            self.field_drawing = None

    # ------------------------------------------------

    def set_agent_positions(self, info):
        for i, drone in enumerate(self.all_drones):
            if i < self.num_robots:
                pos = info[f'robot{i}']['position']
                pos = [p / self.scaling_factor for p in pos] + [self.height]
                sim.setObjectPosition(drone, -1, pos)
                sim.setObjectInt32Param(
                    drone, sim.objintparam_visibility_layer, 1
                )
            else:
                # hide unused drones
                sim.setModelProperty(
                    drone,
                    sim.modelproperty_not_visible
                    | sim.modelproperty_not_collidable
                    | sim.modelproperty_not_detectable
                    | sim.modelproperty_not_dynamic
                )
    
    def set_weed_locations(self):
        weed_obj = sim.getObject('/weed')
        for i, loc in enumerate(self.weed_locations):
            x = [xi/self.scaling_factor for xi in loc]
            new_pos = x + [0]
            object_handle = sim.getObject(f'/weed[{i}]', {'noError': True})
            if object_handle == -1:
                new_weed_obj = sim.copyPasteObjects([weed_obj])[0]
                sim.setObjectPosition(new_weed_obj, -1, new_pos)

    # ------------------------------------------------
    def move_agents(self, info, action):
        """
        action[i] = [dx, dy, spray]
        spray in [0,1]
        """

        for i in range(self.num_robots):
            # ---- move target (position control) ----
            target = sim.getObject(f"/target[{i}]")
            pos = info[f'robot{i}']['position']
            # print("position", pos)
            pos = [p/self.scaling_factor for p in pos] + [self.height]
            sim.setObjectPosition(target, -1, pos)

            # ---- spray control via Lua ----
            spray = float(action[i][2])
            enable = spray > 0.01

            sim.callScriptFunction(
                "setSprayCommand",
                self.nozzle_scripts[i],
                enable,
                spray
            )
    def stop_spraying(self):
        for i in range(self.num_robots):
            sim.callScriptFunction(
                    "setSprayCommand",
                    self.nozzle_scripts[i],
                    True,
                    0
                )


In [6]:
assert False, "Please open the scene file '\simulation\sim_envs\coppeliasim_scene_for_spraying.ttt' before running the following cells"

AssertionError: Please open the scene file '\simulation\sim_envs\coppeliasim_scene_for_spraying.ttt' before running the following cells

Load trained network:

In [7]:
# Load trained network
json_dict = load_experiment_dict_json(json_path)
model = model_loader.load(trained_model_path)
height = 0.35

# Create Gym environment
env = gym.make(
    'MultiRobotEnv-v0',
    field_info=json_dict[f"set{exp_set}"],
    render_mode='human',
    num_robots=num_robots
)
env.metadata['render_fps'] = 30
obs, info = env.reset()
env.render()

# Create simulator
# Make the simulator object, draw the field, and set agent positions
drone_simulator = Drone_simulator(gym_env=env, scaling_factor=5, height=height, num_robots=num_robots)
# drone_simulator.start_simulation()
drone_simulator.draw_field()
drone_simulator.set_agent_positions(info=info)
# drone_simulator.set_weed_locations()
print(info)

c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


{'robot0': {'position': array([18.3, 32.1]), 'capacity': np.float32(10.0)}, 'robot1': {'position': array([39.5, 22.4]), 'capacity': np.float32(12.0)}}


Start simulation!

In [8]:
# Start CoppeliaSim
drone_simulator.start_simulation()
drone_simulator.set_weed_locations()

terminated = False
truncated = False
total_rewards = 0

# -----------------------------
while True:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()
    total_rewards += reward
    print(
        f"reward={reward:.2f}, "
        f"total={total_rewards:.2f}, "
        f"spray={action[:,2]}"
    )
    drone_simulator.move_agents(info, action)
    if terminated or truncated:
        print("Episode finished")        
        drone_simulator.stop_spraying()
        break
    pygame.event.get()
    pygame.time.wait(500)

Simulation started
reward=-20.00, total=-20.00, spray=[0.53403485 0.9892074 ]
reward=265.96, total=245.96, spray=[0.8778184  0.90663457]
reward=514.61, total=760.57, spray=[0.8213254  0.99189913]
reward=1049.60, total=1810.17, spray=[0.7464695  0.92583084]
reward=1005.86, total=2816.02, spray=[0.97620034 0.9215502 ]
reward=1157.26, total=3973.29, spray=[0.93244946 0.02917758]
reward=1662.71, total=5636.00, spray=[0.9965552  0.24521688]
reward=2831.50, total=8467.50, spray=[0.9388625 0.9408741]
reward=4814.38, total=13281.87, spray=[0.9826854  0.98668677]
reward=3609.52, total=16891.40, spray=[0.7100339  0.96935266]
reward=1415.70, total=18307.09, spray=[0.72326225 0.94212747]
reward=981.29, total=19288.38, spray=[0.6183718 0.9852178]
reward=1575.98, total=20864.36, spray=[0.5666702  0.95775676]
reward=1688.98, total=22553.34, spray=[0.40785906 0.99659264]
reward=1156.18, total=23709.53, spray=[0.24816519 0.9902378 ]
reward=1150.77, total=24860.30, spray=[0.25632524 0.9847825 ]
reward=1

Stop simulation (IMPORTANT! otherwise the scene file will be changed if accidentally saved.)

In [9]:
# -----------------------------
drone_simulator.stop_simulation()
env.close()

Simulation reset completed
